# ================================================
# GOLD LAYER — Business Aggregations
# ================================================

In [0]:
from pyspark.sql.functions import count, avg, sum, round, desc, when, col


### ## # Read from Silver table

In [0]:
silver_df = spark.read.table("nyc_taxi_catalog.silver.silver_taxi_data")

## # Gold Table 1 — Hourly Demand

In [0]:
gold_hourly = silver_df \
    .groupBy("pickup_hour") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration")
    ).orderBy("pickup_hour")

gold_hourly.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_catalog.gold.gold_hourly_demand")
print("✅ Gold Hourly Demand Done!")
display(gold_hourly.limit(5))

## # Gold Table 2 — Vendor Revenue

In [0]:
gold_vendor = silver_df \
    .groupBy("VendorID") \
    .agg(
        count("*").alias("total_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    )

gold_vendor.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_catalog.gold.gold_vendor_revenue")
print("✅ Gold Vendor Revenue Done!")
display(gold_vendor.limit(5))

## # Gold Table 3 — Payment Analysis

In [0]:
gold_payment = silver_df \
    .groupBy("payment_label") \
    .agg(
        count("*").alias("total_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    ).orderBy(desc("total_trips"))

gold_payment.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_catalog.gold.gold_payment_analysis")
print("✅ Gold Payment Analysis Done!")
display(gold_payment.limit(5))

## # Gold Table 4 — Trip Efficiency

In [0]:
gold_efficiency = silver_df \
    .groupBy("trip_category") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration"),
        round(avg("cost_per_mile"), 2).alias("avg_cost_per_mile"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    )

gold_efficiency.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_catalog.gold.gold_trip_efficiency")
print("✅ Gold Trip Efficiency Done!")
display(gold_efficiency)

## # Gold Table 5 — Daily Analysis

In [0]:
gold_daily = silver_df \
    .withColumn("day_name",
        when(col("pickup_day") == 1, "Sunday")
        .when(col("pickup_day") == 2, "Monday")
        .when(col("pickup_day") == 3, "Tuesday")
        .when(col("pickup_day") == 4, "Wednesday")
        .when(col("pickup_day") == 5, "Thursday")
        .when(col("pickup_day") == 6, "Friday")
        .otherwise("Saturday")) \
    .groupBy("pickup_day", "day_name") \
    .agg(
        count("*").alias("total_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    ).orderBy("pickup_day")

gold_daily.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_catalog.gold.gold_daily_analysis")
print("✅ Gold Daily Analysis Done!")
display(gold_daily)